In [ ]:
import numpy as np
import optuna
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score

from folds import y

models = ['svdd', 'copod', 'gmm', 'lof']
oof_ranks = {m: rankdata(np.load(f'oof_{m}.npy')) / len(y) for m in models}

def objective(trial):

    weights = {m: trial.suggest_float(f'W_{m}', 0.0, 10.0) for m in models}
    sum_w = sum(weights.values())
    
  
    ensemble = np.zeros(len(y))
    for m in models:
        ensemble += oof_ranks[m] * weights[m]
    ensemble /= sum_w
    

    return average_precision_score(y, ensemble)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=1000)

print(f"Best PR-AUC: {study.best_value:.4f}")
print(f"Best weights: {study.best_params}")

Best PR-AUC: 0.7779
Best weights: {'W_svdd': 8.350346677869501, 'W_copod': 9.55591088508725, 'W_gmm': 1.9154357710367507, 'W_lof': 5.026707796982597}


: 